<a href="https://colab.research.google.com/github/AsyrofiAnam/DeepLearning/blob/main/Sistem_Pembuatan_Kuis_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain langchain-community huggingface_hub sentence-transformers faiss-cpu google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client langchain-google-genai pypdf python-dotenv gradio

In [2]:
import os
import shutil
from dotenv import load_dotenv

shutil.move("ha.env", ".env")

load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")

from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.llms import HuggingFaceHub
from langchain_google_genai import ChatGoogleGenerativeAI

import faiss

In [3]:
from google.colab import files
uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]

loader = PyPDFLoader(pdf_path)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
docs = text_splitter.split_documents(documents)

print(f"Jumlah chunk: {len(docs)}")
print(docs[0].page_content[:300])

Saving Dataset NLP.pdf to Dataset NLP.pdf
Jumlah chunk: 9
Sejarah Indonesia: Dari Masa Prasejarah hingga Kemerdekaan 
Indonesia adalah negara kepulauan yang kaya akan sejarah dan budaya. 
Sejarahnya mencakup berbagai periode penting, mulai dari masa 
prasejarah, kerajaan-kerajaan kuno, masa kolonial, hingga perjuangan 
kemerdekaan. 
1. Masa Prasejarah 
Mas


In [4]:
embedding_model_name = "sentence-transformers/bert-base-nli-max-tokens"
embedding = HuggingFaceEmbeddings(model_name=embedding_model_name)

vector_store = FAISS.from_documents(docs, embedding)
print("Vector store berhasil dibuat.")

<ipython-input-4-3034b62c1258>:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name=embedding_model_name)
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.wa

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.79k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector store berhasil dibuat.


In [5]:
# Buat LLM pipeline dengan model Gemini-1.5-flash
llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    model_kwargs={"temperature": 0, "max_output_tokens": None},
)

/usr/local/lib/python3.11/dist-packages/IPython/core/interactiveshell.py:3553: UserWarning: Parameters {'temperature', 'max_output_tokens'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [6]:
from langchain_core.messages import HumanMessage

def generate_question(context: str) -> str:
    prompt_text = f"""
Buatlah soal kuis **pilihan ganda** berdasarkan teks berikut:

\"\"\"{context}\"\"\"

Format soal yang dihasilkan harus seperti ini:
Pertanyaan: <Isi pertanyaan>
A. <Pilihan A>
B. <Pilihan B>
C. <Pilihan C>
D. <Pilihan D>
Jawaban yang benar: <A/B/C/D>

Pastikan hanya berdasarkan teks yang diberikan. Jangan gunakan informasi dari luar konteks.
Soal:
"""
    messages = [HumanMessage(content=prompt_text)]
    response = llm.invoke(messages)
    return response.content.strip()

In [7]:
import re

def extract_answer_letter(text):
    match = re.search(r'Jawaban yang benar:\s*([A-D])', text)
    return match.group(1).strip() if match else ""

def extract_answer_option(text, letter):
    match = re.search(rf"{letter}\.\s*(.*)", text)
    return match.group(1).strip() if match else ""

def get_short_excerpt(text, answer, window=2):
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    for i, sentence in enumerate(sentences):
        if answer.lower() in sentence.lower():
            start, end = max(0, i - window), min(len(sentences), i + window + 1)
            return ' '.join(sentences[start:end])
    return ' '.join(sentences[:2])

def create_quiz_from_query(query: str, k: int = 5):
    relevant_docs = vector_store.similarity_search(query, k=k*2)  # cari lebih banyak kalau ada soal ganda
    print(f"Menampilkan {k} soal berdasarkan topik: '{query}'\n")

    soal_ke = 1

    for doc in relevant_docs:
        questions_output = generate_question(doc.page_content)

        # Pisahkan jika ada lebih dari satu pertanyaan
        individual_questions = re.split(r'\n(?=Pertanyaan:)', questions_output.strip())

        for q in individual_questions:
            if soal_ke > k:
                break

            letter = extract_answer_letter(q)
            answer = extract_answer_option(q, letter)
            excerpt = get_short_excerpt(doc.page_content, answer)

            print(f"📘 Soal ke-{soal_ke}:\n{q}\n")
            print(f"📌 Alasan Jawaban:\n\"{excerpt}\"\n")
            print("="*80 + "\n")

            soal_ke += 1

In [8]:
create_quiz_from_query("Sejarah Indonesia", k=5)

Menampilkan 5 soal berdasarkan topik: 'Sejarah Indonesia'

📘 Soal ke-1:
Pertanyaan: Menurut teks, kapan masa prasejarah Indonesia dimulai?
A. Setelah kemerdekaan Indonesia
B. Pada masa kerajaan-kerajaan kuno
C. Sekitar 1,5 juta tahun yang lalu
D. Pada masa kolonial
Jawaban yang benar: C

📌 Alasan Jawaban:
"Sejarahnya mencakup berbagai periode penting, mulai dari masa 
prasejarah, kerajaan-kerajaan kuno, masa kolonial, hingga perjuangan 
kemerdekaan. 1. Masa Prasejarah 
Masa prasejarah Indonesia dimulai sejak manusia purba pertama kali 
mendiami nusantara sekitar 1,5 juta tahun yang lalu. Bukti arkeologis"


📘 Soal ke-2:
Pertanyaan: VOC didirikan oleh bangsa manakah dan pada abad ke berapa?
A. Inggris, abad ke-18
B. Portugis, abad ke-17
C. Belanda, abad ke-17
D. Spanyol, abad ke-18
Jawaban yang benar: C


📌 Alasan Jawaban:
"• VOC (Vereenigde Oostindische Compagnie) didirikan Belanda 
pada awal abad ke-17, menjadi perusahaan dagang terbesar dan 
menjalankan kekuasaan politik di Indonesia

In [9]:
import gradio as gr

def process_pdf(pdf_file, query, k=3):
    loader = PyPDFLoader(pdf_file.name)
    docs = loader.load()
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
    chunks = splitter.split_documents(docs)

    vectorstore = FAISS.from_documents(chunks, embedding)
    results = vectorstore.similarity_search(query, k=k)

    quiz_output = ""
    for i, doc in enumerate(results):
        question = generate_question(doc.page_content)
        letter = extract_answer_letter(question)
        answer = extract_answer_option(question, letter)
        excerpt = get_short_excerpt(doc.page_content, answer)

        lines = question.strip().splitlines()
        pertanyaan = [line for line in lines if line.startswith("Pertanyaan:")]

        opsi = []
        seen_letters = set()
        for line in lines:
            match = re.match(r"^([A-D])\.\s", line.strip())
            if match:
                letter = match.group(1)
                if letter not in seen_letters:
                    opsi.append(line.strip())
                    seen_letters.add(letter)
                if len(opsi) == 4:
                    break

        jawaban = [line for line in lines if "Jawaban yang benar:" in line]

        formatted = ""
        if pertanyaan:
            formatted += f"📘 Soal ke-{i+1}:\n{pertanyaan[0]}\n"
        for opt in opsi:
            formatted += f"{opt.strip()}\n"
        if jawaban:
            formatted += f"{jawaban[0]}\n"

        quiz_output += formatted + f"\n📌 Alasan Jawaban:\n\"{excerpt}\"\n"
        quiz_output += "="*80 + "\n\n"


    return quiz_output
# Gradio UI
with gr.Blocks() as demo:
    gr.Markdown("## 📚 Generator Kuis Pilihan Ganda")
    with gr.Row():
        pdf_input = gr.File(label="Upload PDF", file_types=[".pdf"])
        query_input = gr.Textbox(label="Masukkan Topik Soal", placeholder="Contoh: Deep Learning")
        k_input = gr.Slider(minimum=1, maximum=10, value=5, step=1, label="Jumlah Soal")
    generate_btn = gr.Button("🔍 Buat Soal")
    output_box = gr.Textbox(label="Hasil Soal dan Jawaban", lines=20)

    generate_btn.click(fn=process_pdf, inputs=[pdf_input, query_input, k_input], outputs=output_box)

# Jalankan
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5bb03a23ac845ef07f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
